# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [2]:
ПРІЗВИЩЕ_ІМЯ = 'Кривичко Назар'     
ГРУПА        = 'ШІ-34'
НОМЕР        = 11       

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [3]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 11   ·   Кривичко Назар   ·   ШІ-34

Ваші вісім стовпців:
   1. Телефон
   2. Примітка
   3. ЛН
   4. Виписка
   5. Ліжко днів
   6. Прокальцитонін
   7. Білок1
   8. Гематокрит2

Ваша пара для аналізу зв’язку: Вік та spO2 без дот

Ваше додаткове питання:
   Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?


In [8]:
import sys
!"{sys.executable}" -m pip install openpyxl
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Примітка,ЛН,Виписка,Ліжко днів,Прокальцитонін,Білок1,Гематокрит2
0,0958469788 дружина Ганна,NaN,ІІІ-IV,2022-03-10 00:00:00,14,0.126,70.0,45.0
1,"0671778486, 0994887663 дружина Оксана","отримував актемра, часті носові кровотечі, втрата нюху",ІІ-ІІІ,2022-03-16 00:00:00,22,NaN,61.8,57.0
2,"0958768045, чоловік Степан 0951784726",часта втрата свідомосі в стаціонарі,ІІ,2022-01-18 00:00:00,20,NaN,80.5,37.0
3,0959819331. дочка Валентина 0960845205,панкреонекроз,ІІІ-IV,2022-01-13 00:00:00,20,NaN,62.6,34.0
4,"0952294866, дочка Дарина 0678360161",NaN,ІІ-ІІІ,2022-03-11 00:00:00,27,NaN,72.0,48.0
5,0959922983 сестра Софія,NaN,ІІ-ІІІ,2021-12-26 00:00:00,9(5 у реан),NaN,73.0,36.0
6,"0998073702, дружина Наталія 0998382226","глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",ІІ-ІІІ,2022-05-11 00:00:00,18,NaN,76.0,44.5
7,"0637258771, син Сергій 0968413301",NaN,І,2022-05-28 00:00:00,10,NaN,88.0,42.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [9]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Телефон         120 non-null    object 
 1   Примітка        45 non-null     object 
 2   ЛН              121 non-null    object 
 3   Виписка         120 non-null    object 
 4   Ліжко днів      120 non-null    object 
 5   Прокальцитонін  46 non-null     float64
 6   Білок1          118 non-null    float64
 7   Гематокрит2     101 non-null    float64
dtypes: float64(3), object(5)
memory usage: 7.7+ KB


In [10]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [13]:
ПАСПОРТ = {
    'Телефон': (
        'номінальна',
        'це ідентифікатор-мітка, а не величина: значення можна лише порівнювати на рівність, '
        'жодного порядку чи відстані між номерами не існує'),

    'Примітка': (
        'номінальна',
        'вільний текстовий опис супутніх станів — набуває сенсу шкали лише після кодування '
        'у набір бінарних ознак «є/немає», бо порядку між діагнозами немає'),

    'ЛН': (
        'порядкова',
        'ступені легеневої недостатності впорядковані за тяжкістю (І < ІІ < ІІІ < IV), '
        'але відстань між сусідніми ступенями не визначена й не однакова'),

    'Виписка': (
        'інтервальна',
        'дата має сенсовну різницю (скільки днів між виписками), проте нуль календаря умовний, '
        'тому відношення двох дат змісту не мають'),

    'Ліжко днів': (
        'відношень',
        'це підрахунок днів із природним нулем, тому коректні і різниці, і твердження '
        '«вдвічі довше перебування»'),

    'Прокальцитонін': (
        'відношень',
        'концентрація в нг/мл з абсолютним нулем (повна відсутність речовини), '
        'тож відношення значень інтерпретуються як «втричі вищий рівень»'),

    'Білок1': (
        'відношень',
        'концентрація загального білка в г/л з абсолютним нулем, тому допустимі всі '
        'арифметичні дії, зокрема відношення'),

    'Гематокрит2': (
        'відношень',
        'частка об’єму крові, яку займають еритроцити (%), з фізичним нулем — '
        'відношення значень мають зміст, попри обмеженість шкали зверху'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,номінальна,"це ідентифікатор-мітка, а не величина: значення можна ли..."
1,Примітка,номінальна,вільний текстовий опис супутніх станів — набуває сенсу ш...
2,ЛН,порядкова,ступені легеневої недостатності впорядковані за тяжкістю...
3,Виписка,інтервальна,"дата має сенсовну різницю (скільки днів між виписками), ..."
4,Ліжко днів,відношень,"це підрахунок днів із природним нулем, тому коректні і р..."
5,Прокальцитонін,відношень,концентрація в нг/мл з абсолютним нулем (повна відсутніс...
6,Білок1,відношень,"концентрація загального білка в г/л з абсолютним нулем, ..."
7,Гематокрит2,відношень,"частка об’єму крові, яку займають еритроцити (%), з фізи..."


In [14]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [16]:
ПОРЯДОК_ЛН = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']   # звірте з df['ЛН'].unique()

РІВЕНЬ = {'ідентифікатор': 0, 'номінальна': 1, 'порядкова': 2,
          'інтервальна': 3, 'відношень': 4, 'абсолютна': 4}


def рівень_шкали(шкала):
    """'номінальна (неструктурований текст)' -> 1"""
    ш = шкала.strip().lower()
    for назва, р in РІВЕНЬ.items():
        if ш.startswith(назва):
            return р
    raise ValueError(f'невідома шкала: {шкала!r}')


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    рів = рівень_шкали(шкала)

    if рів == 0:
        return р

    # --- мода й частоти: дозволені на всіх шкалах ---
    м = з.mode()
    р['мода'] = list(м) if len(м) <= 3 else f'{len(м)} значень порівну'
    р['частота моди'] = int((з == м.iloc[0]).sum()) if len(м) else 0
    р['топ-3 частоти'] = з.value_counts().head(3).to_dict()

    if р['унікальних'] == р['непорожніх']:
        р['примітка'] = 'усі значення унікальні — частоти нічого не узагальнюють'

    # --- порядкова: медіана та квантилі на рангах ---
    if рів == 2:
        к = pd.Categorical(з, categories=ПОРЯДОК_ЛН, ordered=True).codes
        к = pd.Series(к[к >= 0])                    
        ранг = lambda q: ПОРЯДОК_ЛН[int(np.ceil(к.quantile(q)))]
        р['медіана'] = ранг(0.50)
        р['Q1 / Q3'] = (ранг(0.25), ранг(0.75))
        return р

    # --- інтервальна (дати): середнє і стандартне відхилення ---
    if рів == 3:
        д = pd.to_datetime(з, errors='coerce').dropna()
        р['медіана'] = д.median()
        р['Q1 / Q3'] = (д.quantile(0.25), д.quantile(0.75))
        р['середнє'] = д.mean()
        р['ст. відхилення'] = д.std()
        р['КВ'] = 'не визначений — нуль шкали умовний'
        return р

    # --- відношень / абсолютна: усе ---
    ч = pd.to_numeric(з, errors='coerce').dropna()
    р['медіана'] = round(ч.median(), 3)
    р['Q1 / Q3'] = (round(ч.quantile(0.25), 3), round(ч.quantile(0.75), 3))
    р['середнє'] = round(ч.mean(), 3)
    р['ст. відхилення'] = round(ч.std(ddof=1), 3)
    р['КВ, %'] = round(100 * ч.std(ddof=1) / ч.mean(), 1) if ч.mean() else None
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    for к, v in допустима_статистика(c, ПАСПОРТ[c][0]).items():
        print(f'    {к:<16} {v}')
    print()

--- Телефон  (номінальна)
    непорожніх       120
    унікальних       120
    мода             120 значень порівну
    частота моди     1
    топ-3 частоти    {'0958469788 дружина Ганна': 1, '0671778486, 0994887663 дружина Оксана': 1, '0958768045, чоловік Степан 0951784726': 1}
    примітка         усі значення унікальні — частоти нічого не узагальнюють
    медіана          681960278.0
    Q1 / Q3          (np.float64(636058581.5), np.float64(965931341.5))
    середнє          758798950.391
    ст. відхилення   186311157.182
    КВ, %            24.6

--- Примітка  (номінальна)
    непорожніх       45
    унікальних       44
    мода             ['Поліалергетик']
    частота моди     2
    топ-3 частоти    {'Поліалергетик': 2, 'отримував актемра, часті носові кровотечі, втрата нюху': 1, 'панкреонекроз': 1}
    медіана          nan
    Q1 / Q3          (np.float64(nan), np.float64(nan))
    середнє          nan
    ст. відхилення   nan
    КВ, %            nan

--- ЛН  (порядкова)
   

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [17]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
Білок1,118.0,NaN,NaN,NaN,66.75339,8.37849,50.0,60.0,66.0,72.0,88.0
Гематокрит2,101.0,NaN,NaN,NaN,41.140594,7.565992,16.0,36.0,40.0,46.0,57.0


In [18]:
# --- обчислення "безглуздих" статистик ---
тел = pd.to_numeric(df['Телефон'], errors='coerce').dropna()
СЕР_ТЕЛ = тел.mean()                                 

коди = pd.Categorical(df['ЛН'], categories=ПОРЯДОК_ЛН, ordered=True).codes
СЕР_ЛН = коди[коди >= 0].mean()                       

нс = pd.to_datetime(df['Виписка'], errors='coerce').dropna().astype('int64')
КВ_ДАТИ = 100 * нс.std() / нс.mean()                 

пкт = df['Прокальцитонін'].dropna()
СЕР_ПКТ = пкт.mean()                                   

ТОП_ЛІЖКО = df['Ліжко днів'].mode().iloc[0]         


БЕЗГЛУЗДІ = [
    ('Телефон', 'середнє / медіана / КВ', round(СЕР_ТЕЛ, 2),
     'номер — мітка, а не величина: «середній телефон» не існує як абонент, бо '
     'впорядкованість цифр відображає нумерацію операторів, а не властивість пацієнта; '
     'до того ж число пораховане лише по 23 записах із 120, решта 97 містять текст '
     '(«0958469788 дружина Ганна») і мовчки відкинуті'),

    ('ЛН', 'середній ступінь', round(СЕР_ЛН, 3),
     'значення 2.57 потрапляє між категоріями «ІІ» та «ІІ-ІІІ» і не відповідає жодному '
     'реальному ступеню; усереднення вимагає рівних відстаней між градаціями, а клінічна '
     'відстань між І та ІІ не дорівнює відстані між ІІІ та IV'),

    ('Виписка', 'коефіцієнт варіації', f'{КВ_ДАТИ:.3f} %',
     'КВ — це відношення до середнього, а середня дата (29.01.2022) відлічена від довільної '
     'точки 01.01.1970; змінивши епоху відліку, отримаємо інший КВ для тих самих даних, '
     'тому 0.576 % описує календар, а не розкид виписок — осмислене тут лише '
     'стандартне відхилення у днях (109 днів)'),

    ('Прокальцитонін', 'середнє арифметичне', round(СЕР_ПКТ, 3),
     'шкала дозволяє середнє, але як «типове значення» воно хибне: при медіані 0.1 і Q3 = 0.12 '
     'середнє 0.753 перевищено лише у 3 пацієнтів із 46 — його тягне одне значення 18.22; '
     'КВ = 400 % прямо сигналізує, що розподіл різко асиметричний і центр треба описувати медіаною'),

    ('Ліжко днів', 'top / freq (мода як опис)', f'{ТОП_ЛІЖКО} (12 разів)',
     'describe() подав кількісну змінну як номінальну й не порахував середнє взагалі, бо один '
     'запис «9(5 у реан)» зробив увесь стовпець текстовим; «найчастіше значення 10» — артефакт '
     'округлення термінів до круглих чисел, а не характеристика тривалості госпіталізації'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє / медіана / КВ,758798950.39,"номер — мітка, а не величина: «середній телефон» не існу..."
1,ЛН,середній ступінь,2.57,значення 2.57 потрапляє між категоріями «ІІ» та «ІІ-ІІІ»...
2,Виписка,коефіцієнт варіації,0.576 %,"КВ — це відношення до середнього, а середня дата (29.01...."
3,Прокальцитонін,середнє арифметичне,0.753,"шкала дозволяє середнє, але як «типове значення» воно хи..."
4,Ліжко днів,top / freq (мода як опис),10 (12 разів),describe() подав кількісну змінну як номінальну й не пор...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [20]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × spO2 без дот

Вік: шкала — не у вашому списку
spO2 без дот: шкала — не у вашому списку

   Вік  spO2 без дот
0   55          90.0
1   43          83.0
2   54          78.0
3   68          82.0
4   65          80.0
5   66          72.0


In [21]:
from scipy import stats

ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = ('Вік і сатурація виміряні у шкалі відношень з абсолютним нулем, '
                  'тому допустимі всі арифметичні дії й можна оцінювати саме лінійний зв’язок; '
                  'ранговий Спірмен подано як перевірку стійкості до викидів.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

д = df[[A, B]].apply(pd.to_numeric, errors='coerce').dropna()
ч = д[д[B].between(50, 100)]          # 0.9 — помилка введення, а не сатурація

for назва, вб in [('усі дані', д), ('без викиду', ч)]:
    r, pr = stats.pearsonr(вб[A], вб[B])
    ρ, pρ = stats.spearmanr(вб[A], вб[B])
    print(f'{назва:<12} n={len(вб)}   r={r:+.3f} (p={pr:.4f})   ρ={ρ:+.3f} (p={pρ:.4f})')

усі дані     n=120   r=-0.146 (p=0.1107)   ρ=-0.309 (p=0.0006)
без викиду   n=119   r=-0.266 (p=0.0034)   ρ=-0.334 (p=0.0002)


In [26]:
кодЛН = pd.Categorical(df['ЛН'], ПОРЯДОК_ЛН, ordered=True).codes
к = pd.DataFrame({'ЛН': кодЛН, 'spO2': pd.to_numeric(df[B], errors='coerce')}).dropna()
print('Пірсон(ЛН, spO2) =', round(stats.pearsonr(к.ЛН, к.spO2)[0], 3))

інші = {'І': 1, 'І-ІІ': 2, 'ІІ': 4, 'ІІ-ІІІ': 8, 'ІІІ': 16, 'ІІІ-IV': 32}
к2 = к.assign(ЛН=df['ЛН'].map(інші)).dropna()
print('Пірсон при іншому кодуванні =', round(stats.pearsonr(к2.ЛН, к2.spO2)[0], 3))

# Два різні числа на тих самих даних — бо коди 0…5 і 1…32 обидва однаково довільні. Пірсон оперує відхиленнями від середнього в цих вигаданих одиницях, а реальна клінічна відстань між ступенями невідома. Спірмен обидва рази дасть однаковий результат, бо порядок не змінився.

Пірсон(ЛН, spO2) = -0.251
Пірсон при іншому кодуванні = -0.198


# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [27]:
ПІДГОТОВКА = {
    'Телефон': 'не беремо в модель — ідентифікатор, 120 унікальних значень на 120 записів, '
               'тож інформації про пацієнта не несе й лише дає витік ідентичності',
    'Примітка': 'не беремо «як є» — вільний текст; за потреби витягуємо кілька бінарних прапорців '
                '(«є/немає згадки») за ключовими словами, без порядку між ними',
    'ЛН': 'впорядковані цілі коди 0…5 за заданим списком — порядок зберігаємо, '
          'рівних відстаней не припускаємо; для лінійних моделей надійніше one-hot',
    'Виписка': 'переходимо до тривалості (днів від поступлення) — дата як абсолютна точка '
               'моделі не потрібна, а різниця дат уже належить шкалі відношень',
    'Ліжко днів': 'спершу чистимо текстові записи, далі стандартизація; '
                  'логарифмування не потрібне — розподіл помірно скошений',
    'Прокальцитонін': 'log1p, потім стандартизація — розкид від 0.04 до 18.22 при медіані 0.1, '
                      'тому без логарифма один пацієнт домінує над усією шкалою',
    'Білок1': 'стандартизація — приблизно симетричний розподіл у вузькому діапазоні 50…88',
    'Гематокрит2': 'стандартизація; пропуски в 20 записах позначаємо окремим прапорцем, '
                   'бо факт невимірювання сам по собі інформативний',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,номінальна,"не беремо в модель — ідентифікатор, 120 унікальних значе..."
1,Примітка,номінальна,не беремо «як є» — вільний текст; за потреби витягуємо к...
2,ЛН,порядкова,впорядковані цілі коди 0…5 за заданим списком — порядок ...
3,Виписка,інтервальна,переходимо до тривалості (днів від поступлення) — дата я...
4,Ліжко днів,відношень,"спершу чистимо текстові записи, далі стандартизація; лог..."
5,Прокальцитонін,відношень,"log1p, потім стандартизація — розкид від 0.04 до 18.22 п..."
6,Білок1,відношень,стандартизація — приблизно симетричний розподіл у вузько...
7,Гематокрит2,відношень,стандартизація; пропуски в 20 записах позначаємо окремим...


In [29]:
X = pd.DataFrame(index=df.index)


X['ЛН_код'] = pd.Categorical(df['ЛН'], ПОРЯДОК_ЛН, ordered=True).codes.astype(float)
X.loc[X['ЛН_код'] < 0, 'ЛН_код'] = np.nan

п = pd.to_datetime(df['Поступлення'], errors='coerce')
в = pd.to_datetime(df['Виписка'], errors='coerce')
X['Днів_у_лікарні'] = (в - п).dt.days

лог = np.log1p(pd.to_numeric(df['Прокальцитонін'], errors='coerce'))
X['ПКТ_log_z'] = (лог - лог.mean()) / лог.std()
X['ПКТ_виміряно'] = df['Прокальцитонін'].notna().astype(int)

print(X.describe().T[['count', 'mean', 'std', 'min', 'max']].round(2))

                count   mean    std    min     max
ЛН_код          121.0   2.57   1.20   0.00    5.00
Днів_у_лікарні  118.0  18.95  21.42 -29.00  194.00
ПКТ_log_z        46.0  -0.00   1.00  -0.37    5.04
ПКТ_виміряно    121.0   0.38   0.49   0.00    1.00


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [30]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [31]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [32]:
ВИПРАВЛЕННЯ = [
    (1, 'Середнє арифметичне від номінальної змінної «помер/жив», закодованої як 1/2',
     '1.50 — це не «проміжний результат лікування», а лише наслідок того, що померлих 61, '
     'а виживших 60; при кодуванні 0/1 вийшло б 0.50, при 1/10 — 5.47, тобто число описує коди, а не пацієнтів',
     'Подати частоти й частку: летальність 61/121 = 50.4%; середнє для цієї шкали не рахувати взагалі'),

    (2, 'Різні написання однієї категорії взято за різні категорії',
     'nunique() = 4 дає «чотири статі», бо «ж»/«жін» і «ч»/«чол» не злиті; після нормалізації '
     'лишається дві групи: ж = 75 (62%), ч = 46 (38%), тому й частка 46% для «ж» хибна',
     'Привести значення до єдиного вигляду (str.strip().str[0]) і лише потім рахувати частоти'),

    (3, 'Порядкову шкалу впорядковано за абеткою та усереднено коди',
     'sort_index() дає алфавітний порядок, де «важкий» опиняється першим, а «середній» — останнім, '
     'тож «найлегший ступінь» названо навмання; середнє від кодів цього порядку не має клінічного змісту',
     'Задати порядок явно через pd.Categorical(ordered=True), звести дублікати написань '
     '(«сердній», «легкий ») і подати медіану та квантилі замість середнього'),

    (4, 'Число пораховано на даних, непридатних до обчислення',
     'з 121 запису числовими є лише 27 — решта має вигляд «20-30%», «до 17%» і мовчки відкинута '
     'через errors=\'coerce\'; 39.6% описує довільну четвертину вибірки, а не середній обсяг ураження',
     'Розпарсити текстові діапазони в середину інтервалу, звести все до спільних одиниць '
     'і лише тоді рахувати, вказавши фактичне n'),

    (5, 'Середнє пораховано разом із помилкою введення',
     'у стовпці є значення 0.9 — це 90 із загубленою комою, адже сатурація 0.9% несумісна з життям; '
     'один запис зсуває середнє з 82.74 до 82.05 і псує всі подальші оцінки розкиду',
     'Відфільтрувати фізіологічно неможливі значення (0–50), подати медіану 84 як міру центру '
     'й окремо вказати кількість виправлених записів'),

    (6, 'Коефіцієнт варіації застосовано до інтервальної шкали',
     'КВ є відношенням до середнього, а середня дата відлічена від довільної точки 01.01.1970; '
     'при зміні епохи відліку те саме розсіяння дасть інший КВ, тому 0.576% описує календар, а не потік',
     'Від показника відмовитись: однорідність потоку описувати стандартним відхиленням у днях '
     'або гістограмою госпіталізацій за тижнями'),

    (7, 'Висновок суперечить самому обчисленому числу',
     'r = 0.177 не є «близьким до нуля»: серед жінок померли 43 з 75 (57%), серед чоловіків — 18 з 46 (39%); '
     'знак коефіцієнта до того ж залежить від довільного вибору, кого кодувати нулем',
     'Для двох номінальних змінних будувати таблицю спряженості й рахувати χ² та V Крамера '
     '(V = 0.16, p = 0.079) — зв\'язок слабкий і статистично незначущий, що не те саме, що «відсутній»'),

    (8, 'Нормалізація застосована до всього, що має числовий тип',
     'у select_dtypes(\'number\') потрапили ідентифікатор «номер і/х» та сама цільова змінна «помер/жив», '
     'а min-max рахує межі по всій вибірці, тому викид spO2 = 0.9 стискає решту значень майже в точку',
     'Нормалізувати лише відібрані ознаки, виключивши ідентифікатори й ціль, параметри підганяти '
     'на навчальній частині, а за наявності викидів брати стандартизацію'),

    (9, 'Видалення рядків із пропусками без перевірки результату',
     'dropna() за 77 ознаками залишив 0 рядків зі 121, бо жоден пацієнт не має заповненими всі аналізи; '
     'фраза про «повністю заповнений набір» описує порожню таблицю',
     'Спершу відібрати потрібні для моделі стовпці, далі видаляти рядки лише за ними, '
     'а решту пропусків заповнювати з окремим прапорцем «виміряно/ні»'),

    (10, 'Витік цільової змінної в рейтинг ознак',
     'дата розтину існує рівно у 61 померлого і відсутня у всіх виживших, тому r = 1.00 відтворює '
     'не предиктор, а сам факт смерті — ознака фізично недоступна на момент прогнозу',
     'Виключити всі посмертні ознаки до побудови рейтингу й оцінювати важливість лише за даними, '
     'відомими на момент госпіталізації'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Середнє арифметичне від номінальної змінної «помер/жив»,...","1.50 — це не «проміжний результат лікування», а лише нас...",Подати частоти й частку: летальність 61/121 = 50.4%; сер...
1,2,Різні написання однієї категорії взято за різні категорії,"nunique() = 4 дає «чотири статі», бо «ж»/«жін» і «ч»/«чо...",Привести значення до єдиного вигляду (str.strip().str[0]...
2,3,Порядкову шкалу впорядковано за абеткою та усереднено коди,"sort_index() дає алфавітний порядок, де «важкий» опиняєт...","Задати порядок явно через pd.Categorical(ordered=True), ..."
3,4,"Число пораховано на даних, непридатних до обчислення",з 121 запису числовими є лише 27 — решта має вигляд «20-...,"Розпарсити текстові діапазони в середину інтервалу, звес..."
4,5,Середнє пораховано разом із помилкою введення,"у стовпці є значення 0.9 — це 90 із загубленою комою, ад...","Відфільтрувати фізіологічно неможливі значення (0–50), п..."
5,6,Коефіцієнт варіації застосовано до інтервальної шкали,"КВ є відношенням до середнього, а середня дата відлічена...",Від показника відмовитись: однорідність потоку описувати...
6,7,Висновок суперечить самому обчисленому числу,r = 0.177 не є «близьким до нуля»: серед жінок померли 4...,Для двох номінальних змінних будувати таблицю спряженост...
7,8,"Нормалізація застосована до всього, що має числовий тип",у select_dtypes('number') потрапили ідентифікатор «номер...,"Нормалізувати лише відібрані ознаки, виключивши ідентифі..."
8,9,Видалення рядків із пропусками без перевірки результату,"dropna() за 77 ознаками залишив 0 рядків зі 121, бо жоде...","Спершу відібрати потрібні для моделі стовпці, далі видал..."
9,10,Витік цільової змінної в рейтинг ознак,дата розтину існує рівно у 61 померлого і відсутня у всі...,Виключити всі посмертні ознаки до побудови рейтингу й оц...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [36]:
# Розділ 1
print('НЕПРАВИЛЬНО:', round(df['помер/жив'].mean(), 2))

print('ПРАВИЛЬНО :')
print(df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts().to_string())
print('летальність:', f'{100 * (df["помер/жив"] == 1).mean():.1f} %')

# Розділ 3:
print('НЕПРАВИЛЬНО:')
ст = df['ступінь КТ'].value_counts().sort_index()
print('  найлегший —', ст.index[0], '| найважчий —', ст.index[-1])
коди = {н: i for i, н in enumerate(ст.index)}
print('  середній ступінь =', round(df['ступінь КТ'].map(коди).mean(), 2))

ЗВЕДЕННЯ = {'легкий': 'легкий', 'легкий ': 'легкий', 'легка': 'легкий',
            'середній': 'середній', 'сердній': 'середній',
            'серед ступ': 'середній', 'серед ступеня': 'середній',
            'важкий': 'важкий', 'важкий ': 'важкий',
            'вкрай важкий': 'вкрай важкий', 'вкрай важкий ': 'вкрай важкий',
            'критичний': 'критичний'}
ПОРЯДОК = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
ктг = pd.Categorical(df['ступінь КТ'].map(ЗВЕДЕННЯ), ПОРЯДОК, ordered=True)

к = pd.Series(ктг.codes).replace(-1, np.nan).dropna()
ранг = lambda q: ПОРЯДОК[int(np.ceil(к.quantile(q)))]
print('ПРАВИЛЬНО :')
print(pd.Series(ктг).value_counts().reindex(ПОРЯДОК).to_string())
print(f'  медіана = {ранг(.50)} | Q1–Q3 = {ранг(.25)}–{ранг(.75)} | n = {len(к)} зі 121')


# ═══ Розділ 10: витік цілі в рейтинг ознак ═══
ціль = (df['помер/жив'] == 1).astype(int)
print('НЕПРАВИЛЬНО: Дата розтину, r =',
      round(pd.to_datetime(df['Дата розтину'], errors='coerce').astype('int64').corr(ціль), 3))
print('  заповнена у', df['Дата розтину'].notna().sum(), 'пацієнтів; померли —', ціль.sum())

ПОСМЕРТНІ = ['Дата розтину']
X = (df.select_dtypes('number')
     .drop(columns=['помер/жив'] + ПОСМЕРТНІ, errors='ignore')
     .drop(columns=[c for c in df.columns if 'номер' in c.lower()], errors='ignore'))

р = {c: (abs(X[c].corr(ціль)), X[c].notna().sum())
     for c in X.columns if X[c].notna().sum() >= 60 and pd.notna(X[c].corr(ціль))}
print('ПРАВИЛЬНО :')
print(pd.DataFrame(р, index=['|r|', 'n']).T.sort_values('|r|', ascending=False).head(5).round(3))

НЕПРАВИЛЬНО: 1.5
ПРАВИЛЬНО :
помер/жив
помер    61
вижив    60
летальність: 50.4 %
НЕПРАВИЛЬНО:
  найлегший — важкий | найважчий — середній
  середній ступінь = 5.9
ПРАВИЛЬНО :
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3
  медіана = середній | Q1–Q3 = середній–важкий | n = 42 зі 121
НЕПРАВИЛЬНО: Дата розтину, r = 1.0
  заповнена у 61 пацієнтів; померли — 61
ПРАВИЛЬНО :
         |r|      n
Лім2   0.645  104.0
Л2     0.480  104.0
Вік    0.449  121.0
П/Я2   0.395  104.0
АЛАТ1  0.392  115.0


---
# Етап 7. Висновки та індивідуальне питання

In [37]:
ВИСНОВКИ = [
    'Із восьми стовпців варіанта лише три (Прокальцитонін, Білок1, Гематокрит2) належать до шкали '
    'відношень і допускають повний набір описових статистик, тоді як «Телефон» є ідентифікатором і '
    'має бути виключений з аналізу: 120 унікальних значень на 120 непорожніх записів означають, що '
    'ця ознака не містить жодної інформації про пацієнтів.',

    'Шкала визначає не лише допустиму статистику, а й стійкість результату до технічних рішень: '
    'для пари Вік × spO2 один помилковий запис (0.9 замість 90) змінив кореляцію Пірсона з −0.266 '
    'на −0.146 і перевів p-значення з 0.003 у 0.111, тобто єдине значення зі 120 перевернуло висновок '
    'із «зв\'язок є» на «зв\'язку не виявлено», тоді як рангова кореляція Спірмена зрушилася лише з '
    '−0.334 до −0.309.',

    'Повнота даних у наборі розподілена вкрай нерівномірно і сама несе інформацію: Прокальцитонін '
    'виміряно у 46 пацієнтів зі 121, ступінь КТ визначено у 42, а спроба видалити всі рядки з '
    'пропусками за 77 ознаками залишає нуль записів — тому пропуски тут треба позначати прапорцем '
    '«виміряно/ні», а не відкидати, адже саме призначення аналізу корелює з тяжкістю стану.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Із восьми стовпців варіанта лише три (Прокальцитонін, Білок1, Гематокрит2) належать до шкали відношень і допускають повний набір описових статистик, тоді як «Телефон» є ідентифікатором і має бути виключений з аналізу: 120 унікальних значень на 120 непорожніх записів означають, що ця ознака не містить жодної інформації про пацієнтів.
2. Шкала визначає не лише допустиму статистику, а й стійкість результату до технічних рішень: для пари Вік × spO2 один помилковий запис (0.9 замість 90) змінив кореляцію Пірсона з −0.266 на −0.146 і перевів p-значення з 0.003 у 0.111, тобто єдине значення зі 120 перевернуло висновок із «зв'язок є» на «зв'язку не виявлено», тоді як рангова кореляція Спірмена зрушилася лише з −0.334 до −0.309.
3. Повнота даних у наборі розподілена вкрай нерівномірно і сама несе інформацію: Прокальцитонін виміряно у 46 пацієнтів зі 121, ступінь КТ визначено у 42, а спроба видалити всі рядки з пропусками за 77 ознаками залишає нуль записів — тому пропуски тут треба позначати

In [38]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'До перетворення «Вік» належав до шкали відношень: вік має абсолютний нуль (момент народження), '
    'тому коректними були і різниці («старший на 12 років»), і відношення («вдвічі старший»), і весь '
    'набір статистик — середнє 64.7, стандартне відхилення 12.2, коефіцієнт варіації 18.8%. Після '
    'групування у «до 40», «40–60», «понад 60» змінна стала порядковою: категорії впорядковані за '
    'віком, але відстань між ними не визначена й не однакова — проміжок «до 40» охоплює десятиліття, '
    'а «понад 60» у цих даних тягнеться від 61 до 97 років.\n\n'

    'Втрачено інформацію незворотно: зі 121 різного значення залишилося три, а разом із ними зникли '
    'середнє, стандартне відхилення, коефіцієнт варіації та сама можливість рахувати кореляцію '
    'Пірсона — допустимими лишилися мода, медіана й квантилі. Втрачено й чутливість: кореляція '
    'Спірмена між віком і сатурацією (ρ = −0.334) після групування замінюється порівнянням трьох '
    'середніх, а різниця всередині групи «понад 60» між пацієнтом 61 року і пацієнтом 97 років стає '
    'невидимою. Додатково з\'являється залежність від довільного рішення: межі 40 і 60 обрані ззовні, '
    'і зсув порогу на п\'ять років може змінити картину, хоча самі дані не змінилися.\n\n'

    'Здобуто стійкість і придатність до інтерпретації. Групи нечутливі до помилок введення та '
    'крайніх значень: вік 97 років більше не тягне оцінку, бо входить у категорію нарівні з 61-річним. '
    'Зв\'язок стає видимим без припущення про лінійність — середня сатурація падає з 92.0 у групі '
    '«до 40» до 86.3 у «40–60» і 81.0 у «понад 60», причому перший крок різкіший за другий, чого '
    'коефіцієнт кореляції не показує. Результат формулюється мовою клінічних рішень, а не одиниць '
    'виміру, і допускає перевірку критерієм Краскела — Волліса (H = 12.6, p = 0.0018). Водночас '
    'група «до 40» налічує лише трьох пацієнтів, тож її середнє ненадійне — це ціна, яку завжди '
    'платять за огрублення шкали: частина категорій виявляється заповненою надто рідко.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?

До перетворення «Вік» належав до шкали відношень: вік має абсолютний нуль (момент народження), тому коректними були і різниці («старший на 12 років»), і відношення («вдвічі старший»), і весь набір статистик — середнє 64.7, стандартне відхилення 12.2, коефіцієнт варіації 18.8%. Після групування у «до 40», «40–60», «понад 60» змінна стала порядковою: категорії впорядковані за віком, але відстань між ними не визначена й не однакова — проміжок «до 40» охоплює десятиліття, а «понад 60» у цих даних тягнеться від 61 до 97 років.

Втрачено інформацію незворотно: зі 121 різного значення залишилося три, а разом із ними зникли середнє, стандартне відхилення, коефіцієнт варіації та сама можливість рахувати кореляцію Пірсона — допустимими лишилися мода, медіана й квантилі. Втрачено й чутливість: кореляція Спірмена між віком 

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [39]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'код, перевірено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'код, перевірено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Я перевіряв як саме рахуємо Пірсона і Сімсона кореляцію і перепитував ШІ чому саме так юзаємо і чому так сильно міняється кореляція при зміні значення у 10 разів ( одного )'
ПІДТВЕРДЖУЮ = True 
                      

In [40]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 11 готова до захисту.
Кривичко Назар, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.